In [1]:
from utils import prepare_dataframe
from utils import load_embeddings_from_directory_for_df
import pandas as pd
import torch
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler

# Veri hazırlığı
embedding_directory = 'embeddings/GCB_Pooler_embeddings_patch_before_largewithcodedata_Xlimit3label'
df = prepare_dataframe("merged.csv", True)
resulting_label_count = df['labels'].nunique()

# One-hot encoding işlemi
df_encoded_labels = pd.get_dummies(df, columns=['labels'])
df_encoded_repo = pd.get_dummies(df, columns=['full_repo_name'])

# Özellikler ve etiketler
X_repo = df_encoded_repo.drop(columns=['labels', 'patch', 'index'])
X_repo = X_repo.astype(int)

y = df_encoded_labels.drop(columns=['patch', 'full_repo_name', 'index'])
y = y.astype(int)

df['label_tensor'] = [torch.tensor(row, dtype=torch.float32) for row in y.values]
df['repo_name_tensor'] = [torch.tensor(row, dtype=torch.float32) for row in X_repo.values]
df = df.drop(columns=['full_repo_name', 'labels', 'patch'])

# Embedding'leri yükleme
df = load_embeddings_from_directory_for_df(directory_path=embedding_directory, df=df)

# Embedding ve repo_name_tensor'u birleştirme
df['embedding'] = df['embedding'].apply(lambda x: x.numpy() if isinstance(x, torch.Tensor) else x)
df['repo_name_tensor'] = df['repo_name_tensor'].apply(lambda x: x.numpy() if isinstance(x, torch.Tensor) else x)

# Özellikleri birleştirme
X = np.vstack(df.apply(lambda row: np.concatenate([row['embedding'], row['repo_name_tensor']]), axis=1))

# Özellikleri normalize etme
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Label tensor'daki üç elemanı argmax ile tek bir sınıf etiketine çevirme
y = df['label_tensor'].apply(lambda x: x.numpy().argmax() if isinstance(x, torch.Tensor) else x).values

# Veriyi eğitim ve test setlerine ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelleri tanımlıyoruz
models = [
    ("XGBoost", xgb.XGBClassifier(objective="multi:softmax", num_class=3, eval_metric="mlogloss")),
    ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("CatBoost", CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, verbose=0)),
    ("LightGBM", LGBMClassifier(num_class=3)),
    ("AdaBoost", AdaBoostClassifier(n_estimators=100, random_state=42))
]

# Modellerin doğruluk oranlarını saklamak için bir sözlük
accuracies = {}

# Her bir modeli eğitip doğruluğunu hesaplıyoruz
for model_name, model in models:
    model.fit(X_train, y_train)            # Modeli eğit
    y_pred = model.predict(X_test)         # Tahmin yap
    accuracy = accuracy_score(y_test, y_pred)  # Doğruluğu hesapla
    accuracies[model_name] = accuracy       # Doğruluğu sakla
    print(f"Accuracy for {model_name}: {accuracy}")

# Sonuçları görüntülüyoruz
print("All Model Accuracies:", accuracies)


labels
enhancement    37586
bug            32864
clean          17263
performance     5392
refactor        4596
Name: count, dtype: int64
17263
labels
enhancement    17263
bug            17263
clean          17263
Name: count, dtype: int64


  0%|          | 0/51789 [00:00<?, ?it/s]

Accuracy for XGBoost: 0.7870245221085151
Accuracy for RandomForest: 0.7294844564587758
Accuracy for CatBoost: 0.7365321490635258
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.120809 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 196296
[LightGBM] [Info] Number of data points in the train set: 41431, number of used features: 920
[LightGBM] [Info] Start training from score -1.095600
[LightGBM] [Info] Start training from score -1.101755
[LightGBM] [Info] Start training from score -1.098492
Accuracy for LightGBM: 0.7797837420351419
Accuracy for AdaBoost: 0.6693377099826221
All Model Accuracies: {'XGBoost': 0.7870245221085151, 'RandomForest': 0.7294844564587758, 'CatBoost': 0.7365321490635258, 'LightGBM': 0.7797837420351419, 'AdaBoost': 0.6693377099826221}


In [2]:
for k, i in accuracies.items():
    print(f"{k:<15}: {i}")


XGBoost        : 0.7870245221085151
RandomForest   : 0.7294844564587758
CatBoost       : 0.7365321490635258
LightGBM       : 0.7797837420351419
AdaBoost       : 0.6693377099826221
